In [1]:
import sys
import os
from dotenv import load_dotenv
import guidance
from guidance import system, user, assistant, gen
import math
# Setup local pywhyllm development environment in one line
from notebook_setup import setup_local_pywhyllm
project_root = setup_local_pywhyllm()

# Now import the SimpleModelSuggester
#TFM/pywhyllm/pywhyllm/suggesters --> crear variable 

# project_root = os.path.abspath("/home/moleropa/repositories/master/TFM/pywhyllm/pywhyllm/suggesters/")  # Navigate to TFM/pywhyllm/
# sys.path.insert(0, project_root)

from pywhyllm.suggesters.simple_model_suggester import SimpleModelSuggester

from openai import OpenAI
from portkey_ai import createHeaders
from dotenv import load_dotenv
import time
import base64
from IPython.display import display, Image
from pydantic import BaseModel
import json
load_dotenv()


 Local pywhyllm source already in Python path


True

Example using logprobs

In [2]:
azure_model= "gpt-4o-mini" #"GPT-4o-2024-05-13" 
us_base_url = "https://us.aigw.galileo.roche.com/v1"
portkey_headers = createHeaders(config=os.environ["PORTKEY_AZURE_US_CONFIG"])

azure_openai_client = OpenAI(base_url=us_base_url,
            api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
            default_headers=portkey_headers)
# start = time.time()
# response = azure_openai_client.chat.completions.create(
#     messages=[
#         {
#             "role": "user",
#             "content": "What is Generative AI in one sentence?",
#         }
#     ],
#     model=azure_model,
#     temperature=0.1,
#     logprobs=True,
#     top_logprobs=1,
# )
# print(f"Response time: {time.time() - start}s")
# print(f"Response: {response.choices[0].message.content}")
# print(f"Tokens: {response.usage.total_tokens}")
# print()

In [3]:
azure_model= "gpt-4o-mini" #"GPT-4o-2024-05-13" 
us_base_url = "https://us.aigw.galileo.roche.com/v1"

portkey_headers = createHeaders(config=os.environ["PORTKEY_AZURE_US_CONFIG"])
# azure_openai_client = OpenAI(base_url=us_base_url,
#             api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
#             default_headers=portkey_headers)


# Guidance con modelo OpenAI + base_url + headers
model = guidance.models.OpenAI(
    #"GPT-4o-2024-05-13",
    azure_model,
    api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
    base_url=us_base_url,
    default_headers=portkey_headers,
)


In [4]:
# Import hallbayes repository for hallucination risk assessment
hallbayes_path = os.path.abspath("../../hallbayes")
sys.path.insert(0, hallbayes_path)

from scripts.hallucination_toolkit import OpenAIBackend, OpenAIItem, OpenAIPlanner

Structure: Guidance uses context managers (with guidance.user() and with guidance.assistant()) to structure the conversation
Model State: The lm object accumulates the conversation state as you add to it with +=
Generation: Use guidance.gen() instead of the OpenAI completion parameters
Response Access: Access the generated response via lm['response'] using the name specified in gen()
The guidance approach is more declarative and gives you fine-grained control over the conversation flow, making it particularly useful for complex prompting patterns and multi-step reasoning tasks.

In [5]:
# Make sure to pass the llm parameter explicitly
modeler = SimpleModelSuggester(llm=model)
print("SimpleModelSuggester created successfully with local source code")

SimpleModelSuggester created successfully with local source code


## Test pairwise relationships

In [ ]:
result = modeler.suggest_pairwise_relationship("ice cream sales", "shark attacks")


In [8]:
result = modeler.suggest_pairwise_relationship("ice cream sales", "shark attacks", return_logprobs=True)


StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

In [9]:
result

{'result': [None,
  None,
  'To determine the most likely cause-and-effect relationship among the options provided, we need to analyze the nature of the two events: ice cream sales and shark attacks.\n\nA. **Ice cream sales cause shark attacks**: This relationship seems unlikely because the consumption of ice cream does not have a direct influence on shark behavior. Shark attacks are more likely to be affected by factors such as water temperature, visibility, and the presence of swimmers rather than ice cream sales.\n\nB. **Shark attacks cause ice cream sales**: This relationship is also questionable because shark attacks are rare and do not typically lead to an increase in ice cream sales. In fact, the marketing and sales of ice cream are not connected to the occurrence of shark attacks. \n\nC. **Neither ice cream sales nor shark attacks cause each other**: This option suggests that there is no causal relationship between the two events. This seems to be the most reasonable conclusion

In [ ]:
# # doesn't work
# if result[0] is not None:
#     print(f"{result[0]} causes {result[1]}")
# else:
#     print(f"neither causes the other")

## Let's build a graph among our Use case Variables - Diabetes Mellitus 

In [ ]:
# variables = ["ice cream sales", "temperature", "cavities"]
# results = modeler.suggest_relationships(variables)

# results

In [6]:
# Simple timeout handling approach
start_time = time.time()

variables = ["bmi", "systolic pressure (sysbp)", "diastolic pressure (diabp)", "age", "cardiac ischemia", "diabetes mellitus", "major acute cardiovascular event"]

results = modeler.suggest_relationships(variables)

elapsed_time = time.time() - start_time

1/21.0: Querying for relationship between bmi and systolic pressure (sysbp)


StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

	bmi directly causes systolic pressure (sysbp)
2/21.0: Querying for relationship between bmi and diastolic pressure (diabp)
	bmi directly causes diastolic pressure (diabp)
3/21.0: Querying for relationship between bmi and age
	age directly causes bmi
4/21.0: Querying for relationship between bmi and cardiac ischemia
	bmi directly causes cardiac ischemia
5/21.0: Querying for relationship between bmi and diabetes mellitus
	bmi directly causes diabetes mellitus
6/21.0: Querying for relationship between bmi and major acute cardiovascular event
	bmi directly causes major acute cardiovascular event
7/21.0: Querying for relationship between systolic pressure (sysbp) and diastolic pressure (diabp)
	No direct relationship found between systolic pressure (sysbp) and diastolic pressure (diabp)
8/21.0: Querying for relationship between systolic pressure (sysbp) and age
	age directly causes systolic pressure (sysbp)
9/21.0: Querying for relationship between systolic pressure (sysbp) and cardiac isc

In [8]:
results

{('bmi',
  'systolic pressure (sysbp)'): "To evaluate the cause-and-effect relationship between BMI (body mass index) and systolic pressure (sysbp), let's consider some clinical and biological reasoning:\n\n- BMI is a measure of body fat based on height and weight.\n- Systolic blood pressure is the pressure in your blood vessels when your heart beats.\n\n(A) BMI causes systolic pressure:\nResearch and physiological mechanisms establish that higher BMI (usually associated with overweight/obesity) is a risk factor for elevated blood pressure. Excess body mass increases vascular resistance and blood volume, leading to higher systolic pressure. This is a well-documented relationship in epidemiology and clinical medicine.\n\n(B) Systolic pressure causes BMI:\nThere is little or no evidence to suggest that elevated systolic blood pressure directly causes changes in BMI. While high blood pressure might have downstream effects on health, it does not induce changes in body fat or weight directl

In [22]:
results

{('bmi',
  'systolic pressure (sysbp)'): 'BMI (Body Mass Index) is a measure of body fat based on height and weight, while systolic blood pressure (sysbp) is a measure of the pressure in arteries when the heart beats. Epidemiologically and physiologically, higher BMI is known to be associated with increased risk of hypertension and higher systolic blood pressure due to increased vascular resistance and metabolic demands. It is less plausible biologically that higher systolic pressure would directly cause an increase in BMI. \n\nHowever, both BMI and systolic blood pressure can be influenced by common factors such as age, diabetes, cardiac ischemia, and cardiovascular events. This means the association could be confounded or mediated by these variables. Nevertheless, the direct cause-and-effect relationship is more likely to be BMI causing changes in systolic pressure rather than vice versa.\n\nFinal answer: <answer>A</answer>',
 ('bmi',
  'diastolic pressure (diabp)'): 'Both body mass 

In [23]:
import json

# Convert tuple keys to strings for JSON serialization
results_serializable = {
    f"{cause} -> {effect}": description 
    for (cause, effect), description in results.items()
}

# Save to JSON file
output_path = "causal_relationships_diabetes_baseline.json"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(results_serializable, f, indent=2, ensure_ascii=False)

print(f"Results saved to {output_path}")
print(f"Total relationships: {len(results_serializable)}")

Results saved to causal_relationships_diabetes_baseline.json
Total relationships: 19


In [ ]:
res

In [9]:
results.keys()

dict_keys([('bmi', 'systolic pressure (sysbp)'), ('bmi', 'diastolic pressure (diabp)'), ('age', 'bmi'), ('bmi', 'cardiac ischemia'), ('bmi', 'diabetes mellitus'), ('bmi', 'major acute cardiovascular event'), ('age', 'systolic pressure (sysbp)'), ('systolic pressure (sysbp)', 'cardiac ischemia'), ('diabetes mellitus', 'systolic pressure (sysbp)'), ('systolic pressure (sysbp)', 'major acute cardiovascular event'), ('age', 'diastolic pressure (diabp)'), ('diabetes mellitus', 'diastolic pressure (diabp)'), ('diastolic pressure (diabp)', 'major acute cardiovascular event'), ('age', 'cardiac ischemia'), ('age', 'diabetes mellitus'), ('age', 'major acute cardiovascular event'), ('diabetes mellitus', 'cardiac ischemia'), ('cardiac ischemia', 'major acute cardiovascular event'), ('diabetes mellitus', 'major acute cardiovascular event')])

In [8]:
len(results)

19

## Build Adjacency Matrix from Causal Relationships

In [16]:
from collections import defaultdict, deque

def topological_sort_variables(causal_relationships):
    """
    Ordena las variables en orden topológico basado en las relaciones causales.
    Las variables que son causas aparecerán antes que sus efectos.
    
    Args:
        causal_relationships: dict con claves (causa, efecto)
    
    Returns:
        list: Variables ordenadas topológicamente
    """
    # Extract all variables
    all_vars = set()
    for (cause, effect) in causal_relationships.keys():
        all_vars.add(cause)
        all_vars.add(effect)
    
    # Build adjacency list and compute in-degrees
    graph = defaultdict(list)
    in_degree = {var: 0 for var in all_vars}
    
    for (cause, effect) in causal_relationships.keys():
        graph[cause].append(effect)
        in_degree[effect] += 1
    
    # Kahn's algorithm for topological sorting
    queue = deque([var for var in all_vars if in_degree[var] == 0])
    topo_order = []
    
    while queue:
        # Sort the queue to have deterministic ordering
        queue = deque(sorted(queue))
        var = queue.popleft()
        topo_order.append(var)
        
        for neighbor in sorted(graph[var]):
            in_degree[neighbor] -= 1
            if in_degree[neighbor] == 0:
                queue.append(neighbor)
    
    # Check if there's a cycle
    if len(topo_order) != len(all_vars):
        print("⚠️ WARNING: Cycle detected in causal graph!")
        print("Variables not in topological order:", set(all_vars) - set(topo_order))
        # Add remaining variables in alphabetical order
        remaining = sorted(set(all_vars) - set(topo_order))
        topo_order.extend(remaining)
    
    return topo_order

# Get topologically sorted order
topo_order = topological_sort_variables(results)

In [18]:
def build_adjacency_matrix_ordered(causal_relationships, variable_order):
    """
    Build an adjacency matrix with a specific variable ordering.
    
    Args:
        causal_relationships: dict with keys as (cause, effect) tuples
        variable_order: list of variables in desired order
    
    Returns:
        pandas.DataFrame: Adjacency matrix where entry (i,j) = 1 means j→i (j causes i)
    """
    n = len(variable_order)
    adj_matrix = np.zeros((n, n), dtype=int)
    
    # Create index mapping
    var_to_idx = {var: idx for idx, var in enumerate(variable_order)}
    
    # Fill in the edges
    for (cause, effect) in causal_relationships.keys():
        cause_idx = var_to_idx[cause]
        effect_idx = var_to_idx[effect]
        adj_matrix[effect_idx, cause_idx] = 1  # Row = effect, Column = cause
    
    df = pd.DataFrame(adj_matrix, index=variable_order, columns=variable_order)
    
    return df

adj_matrix_ordered = build_adjacency_matrix_ordered(results, topo_order)

# Check if it's lower triangular (all elements above diagonal should be 0)
upper_triangle_sum = np.triu(adj_matrix_ordered.values, k=1).sum()
if upper_triangle_sum == 0:
    print("Matrix is lower triangular (correct topological order)")
else:
    print("Matrix is NOT lower triangular (possible cycle or ordering issue)")

Matrix is lower triangular (correct topological order)


In [19]:
adj_matrix_ordered

,age,bmi,diabetes mellitus,diastolic pressure (diabp),systolic pressure (sysbp),cardiac ischemia,major acute cardiovascular event
age,0,0,0,0,0,0,0
bmi,1,0,0,0,0,0,0
diabetes mellitus,1,1,0,0,0,0,0
diastolic pressure (diabp),1,1,1,0,0,0,0
systolic pressure (sysbp),1,1,1,0,0,0,0
cardiac ischemia,1,1,1,0,1,0,0
major acute cardiovascular event,1,1,1,1,1,1,0


In [20]:
# Analyze the structure with the new ordering

for i, var in enumerate(topo_order):
    # Get causes of this variable
    causes = adj_matrix_ordered.columns[adj_matrix_ordered.loc[var] == 1].tolist()
    # Get effects of this variable
    effects = adj_matrix_ordered.index[adj_matrix_ordered[var] == 1].tolist()
    
    print(f"\n{i+1}. {var}:")
    if causes:
        print(f"   Caused by: {', '.join(causes)}")
    else:
        print(f"   ROOT CAUSE (no incoming edges)")
    
    if effects:
        print(f"   Causes: {', '.join(effects)}")
    else:
        print(f"   TERMINAL NODE (no outgoing edges)")


1. age:
   ROOT CAUSE (no incoming edges)
   Causes: bmi, diabetes mellitus, diastolic pressure (diabp), systolic pressure (sysbp), cardiac ischemia, major acute cardiovascular event

2. bmi:
   Caused by: age
   Causes: diabetes mellitus, diastolic pressure (diabp), systolic pressure (sysbp), cardiac ischemia, major acute cardiovascular event

3. diabetes mellitus:
   Caused by: age, bmi
   Causes: diastolic pressure (diabp), systolic pressure (sysbp), cardiac ischemia, major acute cardiovascular event

4. diastolic pressure (diabp):
   Caused by: age, bmi, diabetes mellitus
   Causes: major acute cardiovascular event

5. systolic pressure (sysbp):
   Caused by: age, bmi, diabetes mellitus
   Causes: cardiac ischemia, major acute cardiovascular event

6. cardiac ischemia:
   Caused by: age, bmi, diabetes mellitus, systolic pressure (sysbp)
   Causes: major acute cardiovascular event

7. major acute cardiovascular event:
   Caused by: age, bmi, diabetes mellitus, diastolic pressure (d

## Variable Mapping: Conceptual to Dataset Features

Mapping between conceptual causal variables and actual diabetes dataset features:

In [16]:
# Mapping conceptual variables to actual dataset features
variable_mapping = {
    "obesity": {
        "primary": "obese275",  # Binary obesity indicator (BMI >= 27.5)
        "related": ["bmi"],     # Continuous BMI measure
        "description": "Obesity status based on BMI threshold of 27.5"
    },
    
    "hypertension": {
        "primary": "hyperten",  # Hypertension diagnosis
        "related": ["sysbp", "diabp", "timehyp"],  # Blood pressure measures and time to hypertension
        "description": "Hypertension status with systolic/diastolic BP"
    },
    
    "age": {
        "primary": "age",       # Age in years
        "related": ["age1"],    # Possibly age-related transformation
        "description": "Patient age"
    },
    
    "cardiac ischemia": {
        "primary": "mifchd",    # Myocardial infarction or fatal CHD
        "related": ["hospmi", "angina", "anychd", "timemi", "timemifc", "timechd"],
        "description": "Cardiac ischemic events (MI, angina, CHD)"
    },
    
    "diabetes mellitus": {
        "primary": "diabetes1", # Diabetes diagnosis
        "related": ["glucose"], # Blood glucose level
        "description": "Diabetes status with glucose measurements"
    },
    
    "major acute cardiovascular event": {
        "primary": "cvd",       # Cardiovascular disease event
        "related": ["death", "stroke", "timestrk", "timecvd", "timedth"],
        "description": "Major cardiovascular events (CVD, stroke, death)"
    }
}

# Additional relevant features for causal analysis
additional_features = {
    "confounders": ["sex1", "cursmoke1", "cigpday", "totchol", "chol1"],
    "outcome_timing": ["timeap", "timemi", "timemifc", "timechd", "timestrk", "timecvd", "timedth", "timehyp"],
    "identifiers": ["randid"]
}



for concept, mapping in variable_mapping.items():
    print(f"\n{concept.upper()}")
    print(f"   Primary feature: {mapping['primary']}")
    print(f"   Description: {mapping['description']}")
    if mapping['related']:
        print(f"   Related features: {', '.join(mapping['related'])}")

print("\n" + "=" * 70)
print("ADDITIONAL FEATURES FOR CAUSAL ANALYSIS")
print("=" * 70)
print(f"Confounders: {', '.join(additional_features['confounders'])}")
print(f"Outcome timing: {len(additional_features['outcome_timing'])} time-to-event variables")

# Create lists for the causal discovery analysis
conceptual_variables = list(variable_mapping.keys())
primary_features = [mapping['primary'] for mapping in variable_mapping.values()]

print("\n" + "=" * 70)
print(f"Conceptual variables for LLM: {conceptual_variables}")
print(f"Primary dataset features: {primary_features}")
print("=" * 70)


OBESITY
   Primary feature: obese275
   Description: Obesity status based on BMI threshold of 27.5
   Related features: bmi

HYPERTENSION
   Primary feature: hyperten
   Description: Hypertension status with systolic/diastolic BP
   Related features: sysbp, diabp, timehyp

AGE
   Primary feature: age
   Description: Patient age
   Related features: age1

CARDIAC ISCHEMIA
   Primary feature: mifchd
   Description: Cardiac ischemic events (MI, angina, CHD)
   Related features: hospmi, angina, anychd, timemi, timemifc, timechd

DIABETES MELLITUS
   Primary feature: diabetes1
   Description: Diabetes status with glucose measurements
   Related features: glucose

MAJOR ACUTE CARDIOVASCULAR EVENT
   Primary feature: cvd
   Description: Major cardiovascular events (CVD, stroke, death)
   Related features: death, stroke, timestrk, timecvd, timedth

ADDITIONAL FEATURES FOR CAUSAL ANALYSIS
Confounders: sex1, cursmoke1, cigpday, totchol, chol1
Outcome timing: 8 time-to-event variables

Conceptua

### Key Insights:

1. **Obesity**: Use `obese275` (binary) or `bmi` (continuous)
2. **Hypertension**: Use `hyperten` with `sysbp`/`diabp` as supporting variables
3. **Age**: Direct mapping to `age`
4. **Cardiac Ischemia**: `mifchd` is the primary indicator, with `angina`, `hospmi`, `anychd` as related events
5. **Diabetes Mellitus**: `diabetes1` with `glucose` as biomarker
6. **Major Acute Cardiovascular Event**: `cvd` captures major events, with `death` and `stroke` as specific outcomes

**Important confounders to include**: `sex1`, `cursmoke1`, `cigpday`, `totchol` for proper causal analysis.

Results already in the form cause effect

In [17]:
results

{('obesity',
  'hypertension'): 'Obesity is a well-established risk factor for hypertension (high blood pressure). Increased body fat can lead to changes in the cardiovascular system, such as increased blood volume, higher cardiac output, and vascular resistance, all of which contribute to high blood pressure. Numerous epidemiological studies demonstrate that people with obesity are much more likely to develop hypertension.\n\nOn the contrary, there is little evidence suggesting that hypertension directly causes obesity. While there may be shared risk factors (such as poor diet and physical inactivity), hypertension itself does not cause significant weight gain. Therefore, option B is less likely.\n\nOption C is also unlikely because the causal relationship from obesity to hypertension is widely documented.\n\nThus, the most likely cause-and-effect relationship is:\n\n<answer>A</answer>',
 ('age',
  'obesity'): 'Let’s examine each option:\n\nA. Obesity causes age.  \nThis implies that 

In [11]:
results[('obesity', 'hypertension')]

'Obesity is a well-established risk factor for developing hypertension (high blood pressure). Excess body fat leads to various physiological changes, such as increased blood volume, activation of the renin-angiotensin-aldosterone system, insulin resistance, and inflammation, all of which contribute to raised blood pressure.\n\nWhile hypertension itself rarely causes obesity, the presence of hypertension does not typically lead to significant weight gain, nor is weight gain a direct consequence of having high blood pressure.\n\nTherefore, based on epidemiological studies and mechanistic research, the cause-and-effect relationship where obesity causes hypertension is much more likely.\n\n<answer>A</answer>'

## Confounders

In [4]:
# variables = ["ice cream sales", "temperature", "cavities"]
variables = ["obesity", "hypertension", "age", "cardiac ischemia", "diabetes mellitus", "major acute cardiovascular event"]

latents = modeler.suggest_confounders(variables, "diabetes mellitus", "cardiac ischemia")

print(latents)

StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

['obesity', 'hypertension', 'age']


In [ ]:
# latents = modeler.suggest_confounders(["weight", "diet", "age"], "vitamin c", "cardiovascular health")

# print(latents)

In [4]:
variables = ["obesity", "hypertension", "age", "cardiac ischemia", "diabetes mellitus", "major acute cardiovascular event"]


confounders = modeler.suggest_confounders_custom(variables, "diabetes mellitus", "cardiac ischemia")

print(confounders)

StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

['obesity', 'hypertension', 'age']


it is identifying the confounders correctly if we consider the ground truth our paper Kyriacou

In [6]:
variables = ["obesity", "hypertension", "age", "cardiac ischemia", "diabetes mellitus", "major acute cardiovascular event"]

colliders = modeler.suggest_colliders_custom(variables, "diabetes mellitus", "cardiac ischemia")   
print(colliders)

StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

['hypertension', 'major acute cardiovascular event', 'hypertension', 'major acute cardiovascular event']


## Identification support

### Instrumental variables

In [ ]:
from pywhyllm.suggesters.simple_identification_suggester import SimpleIdentificationSuggester
identifier = SimpleIdentificationSuggester('gpt-4o-mini')

In [ ]:
variables = ["cigarette taxes", "rain", "car sales", "property taxes", "heart attacks"]
ivs = identifier.suggest_iv(variables, "smoking", "birth weight")

ivs

### Backdoor variables

In [ ]:
variables = ["Age", "Sex", "HbA1c", "HDL", "LDL", "eGFR", "Prior MI",
             "Prior Stroke or TIA", "Prior Heart Failure", "Cardiovascular medication",
             "T2DM medication", "Insulin", "Morbid obesity", "First occurrence of Nonfatal myocardial infarction, nonfatal stroke, death from all cause",
             "semaglutide treatment", "Semaglutide medication", "income", "musical taste"]

backdoors = identifier.suggest_backdoor(variables,
                            treatment="semaglutide treatment", outcome = "cardiovascular health")

print(backdoors)

### Frontdoor

In [ ]:
frontdoors = identifier.suggest_frontdoor(variables,
                            treatment="semaglutide treatment", outcome = "cardiovascular health")

print(frontdoors)

## Hallucination Risk Assessment for Causal Discovery

Let's assess the reliability of our causal discovery responses using hallbayes toolkit.

In [10]:
# Create hallbayes backend using the existing Azure OpenAI client
# We need to bypass the default OpenAI API key requirement
import os
from hallbayes import OpenAIBackend, OpenAIItem, OpenAIPlanner
temp_api_key = os.environ.get("PORTKEY_AZURE_US_API_KEY")

# Create the backend with our Azure model and then replace the client
hallbayes_backend = OpenAIBackend(model=azure_model, api_key=temp_api_key)

azure_openai_client = OpenAI(base_url=us_base_url,
            api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
            default_headers=portkey_headers)


# Replace the default client with our configured Azure OpenAI client 
hallbayes_backend.client = azure_openai_client


In [11]:
planner = OpenAIPlanner(hallbayes_backend, temperature=0.3)

simplify next cell

### Batch Assessment for Multiple Causal Relationships

Now let's assess multiple causal relationships from your diabetes case study.

In [6]:
# Ejemplo de uso: Validar las relaciones encontradas en tu diabetes case study
variables = ["obesity", "hypertension", "age"]

# 1. Generar relaciones causales
relationships = modeler.suggest_relationships(variables)


1/3.0: Querying for relationship between obesity and hypertension


StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

	obesity causes hypertension
2/3.0: Querying for relationship between obesity and age
	age causes obesity
3/3.0: Querying for relationship between hypertension and age
	age causes hypertension


In [7]:
relationships

{('obesity',
  'hypertension'): "Obesity is a well-established risk factor for hypertension (high blood pressure). Excess body fat can lead to changes in the body's metabolism, increase blood volume, and elevate pressure on arterial walls, all contributing to the development of hypertension. Numerous epidemiological studies support that obesity increases the likelihood of developing hypertension.\n\nOn the other hand, while hypertension can be associated with lifestyle factors that also promote obesity, there is little evidence to suggest that hypertension itself directly causes obesity. Hypertension is more often a consequence rather than a cause in this relationship.\n\nTherefore, the most likely cause-and-effect relationship is:\n\n<answer>A</answer>",
 ('age',
  'obesity'): 'Age causes obesity is more likely. Here’s why: Age is a biological process and everyone ages over time; obesity is a condition that can develop as a result of various factors, including age. As people get older

In [19]:
relationships= modeler.suggest_relationships(["temperature","altitude"])

1/1.0: Querying for relationship between temperature and altitude


StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

	altitude causes temperature


In [20]:
relationships

{('altitude',
  'temperature'): 'The more likely cause-and-effect relationship is B. Altitude affects temperature, not the other way around. As altitude increases (you move higher above sea level), the temperature generally decreases because the atmosphere becomes thinner and is less able to retain heat. There is no physical mechanism by which temperature alone affects the altitude of a location. Thus, the most plausible causal link is that altitude causes changes in temperature.\n\n<answer>B</answer>'}

### Hallbayes validation with closed_book

In [ ]:
# # Unified hallbayes validation with consistency checking
# def validate_relationships_with_consistency(relationships, planner, method="closed_book", threshold=0.10, n_runs=3):
#     """
#     Validates causal relationships using hallbayes with consistency checking
    
#     Args:
#         relationships: dict from suggest_relationships()
#         planner: configured hallbayes planner
#         method: "closed_book" or "with_evidence"
#         threshold: max acceptable hallucination risk
#         n_runs: number of validation runs per relationship (for consistency)
    
#     Returns:
#         dict: validation results with consistency information
#     """
#     print(f"Validating {len(relationships)} relationships using {method.upper()} method")
#     print(f"Running {n_runs} tests per relationship for consistency check")
#     print("=" * 60)
    
#     all_results = {}
    
#     for rel_idx, ((cause, effect), description) in enumerate(relationships.items(), 1):
#         print(f"\n{rel_idx}/{len(relationships)}: {cause} → {effect}")
        
#         # Prepare detailed prompt based on method
#         if method == "closed_book":
#             prompt = f"""
#             Medical Knowledge Assessment:
            
#             Claim: "{cause}" causes "{effect}"
            
#             Question: Based on established medical and scientific knowledge, 
#             is this causal relationship scientifically valid?
            
#             Answer: Yes/No with brief justification (2-3 established mechanisms).
#             """
#         elif method == "with_evidence":
#             prompt = f"""
#             Evidence-Based Assessment:
            
#             Provided Evidence: {description}
            
#             Claim: "{cause}" causes "{effect}"
            
#             Question: Based on the provided evidence and your knowledge, 
#             is this causal relationship valid and well-supported?
            
#             Answer: Yes/No referencing the provided evidence and additional knowledge.
#             """
#         else:
#             raise ValueError(f"Unknown method: {method}")
        
#         # Run multiple validations for consistency
#         run_results = []
#         valid_count = 0
        
#         for run in range(n_runs):
#             try:
#                 item = OpenAIItem(prompt=prompt, n_samples=3, m=4, skeleton_policy=method)
#                 metrics = planner.run([item], h_star=threshold, isr_threshold=1.0)
                
#                 if metrics:
#                     metric = metrics[0]
#                     is_valid = metric.decision_answer
#                     risk = metric.roh_bound
                    
#                     if is_valid:
#                         valid_count += 1
                    
#                     run_results.append({
#                         'valid': is_valid,
#                         'risk': risk,
#                         'run': run + 1
#                     })
                    
#                     status = "VALID" if is_valid else "REJECT"
#                     print(f"  Run {run + 1}: {status} (Risk: {risk:.1%})")
#                 else:
#                     run_results.append({'valid': False, 'risk': 1.0, 'run': run + 1, 'error': 'No metrics'})
#                     print(f"  Run {run + 1}: ERROR (No metrics)")
                    
#             except Exception as e:
#                 run_results.append({'valid': False, 'risk': 1.0, 'run': run + 1, 'error': str(e)})
#                 print(f"  Run {run + 1}: ERROR ({str(e)[:50]}...)")
        
#         # Calculate consistency and final decision
#         consistency_rate = valid_count / n_runs
#         avg_risk = sum(r.get('risk', 1.0) for r in run_results) / len(run_results)
        
#         # Determine final validation (majority vote)
#         final_valid = valid_count > (n_runs / 2)
        
#         # Determine consistency level
#         if consistency_rate == 1.0 or consistency_rate == 0.0:
#             consistency = "CONSISTENT"
#         elif consistency_rate >= 0.66:
#             consistency = "MOSTLY_CONSISTENT"  
#         else:
#             consistency = "INCONSISTENT"
        
#         all_results[(cause, effect)] = {
#             'final_decision': final_valid,
#             'consistency': consistency,
#             'consistency_rate': consistency_rate,
#             'avg_risk': avg_risk,
#             'valid_runs': valid_count,
#             'total_runs': n_runs,
#             'method': method,
#             'runs': run_results,
#             'description': description
#         }
        
#         print(f"  → Final: {'VALID' if final_valid else 'REJECT'} | Consistency: {consistency} ({valid_count}/{n_runs})")
    
#     return all_results

# # Test the unified function with professional prompts
# print("Testing unified validation function with professional prompts:")
# unified_results = validate_relationships_with_consistency(
#     results, planner, method="with_evidence", threshold=0.10, n_runs=3
# )

In [34]:
# Unified hallbayes validation with consistency checking BUENO
def validate_relationships_with_consistency(relationships, planner, method="closed_book", threshold=0.10, n_runs=3):
    """
    Validates causal relationships using hallbayes with consistency checking
    
    Args:
        relationships: dict from suggest_relationships()
        planner: configured hallbayes planner
        method: "closed_book" or "with_evidence"
        threshold: max acceptable hallucination risk
        n_runs: number of validation runs per relationship (for consistency)
    
    Returns:
        dict: validation results with consistency information
    """
    print(f"Validating {len(relationships)} relationships using {method.upper()} method")
    print(f"Running {n_runs} tests per relationship for consistency check")
    print("=" * 60)
    
    all_results = {}
    
    for rel_idx, ((cause, effect), description) in enumerate(relationships.items(), 1):
        print(f"\n{rel_idx}/{len(relationships)}: {cause} → {effect}")
        
        # Prepare detailed prompt based on method
        if method == "closed_book":
            prompt = f"""
            Medical Knowledge Assessment:
            
            Claim: "{cause}" causes "{effect}"
            
            Question: Based on established medical and scientific knowledge, 
            is this causal relationship scientifically valid?
            
            Answer: Yes/No with brief justification (2-3 established mechanisms).
            """
        elif method == "with_evidence":
            prompt = f"""
            Evidence-Based Assessment:
            
            Provided Evidence: {description}
            
            Claim: "{cause}" causes "{effect}"
            
            Question: Based on the provided evidence and your knowledge, 
            is this causal relationship valid and well-supported?
            
            Answer: Yes/No referencing the provided evidence and additional knowledge.
            """
        else:
            raise ValueError(f"Unknown method: {method}")
        
        # Run multiple validations for consistency
        run_results = []
        valid_count = 0
        
        for run in range(n_runs):
            try:
                item = OpenAIItem(prompt=prompt, n_samples=3, m=4, skeleton_policy=method)
                metrics = planner.run([item], h_star=threshold, isr_threshold=1.0)
                
                if metrics:
                    metric = metrics[0]
                    is_valid = metric.decision_answer
                    risk = metric.roh_bound
                    
                    if is_valid:
                        valid_count += 1
                    
                    run_results.append({
                        'valid': is_valid,
                        'risk': risk,
                        'run': run + 1
                    })
                    
                    status = "VALID" if is_valid else "REJECT"
                    print(f"  Run {run + 1}: {status} (Risk: {risk:.1%})")
                else:
                    run_results.append({'valid': False, 'risk': 1.0, 'run': run + 1, 'error': 'No metrics'})
                    print(f"  Run {run + 1}: ERROR (No metrics)")
                    
            except Exception as e:
                run_results.append({'valid': False, 'risk': 1.0, 'run': run + 1, 'error': str(e)})
                print(f"  Run {run + 1}: ERROR ({str(e)[:50]}...)")
        
        # Calculate consistency and final decision
        consistency_rate = valid_count / n_runs
        avg_risk = sum(r.get('risk', 1.0) for r in run_results) / len(run_results)
        
        # Determine final validation (majority vote)
        final_valid = valid_count > (n_runs / 2)
        
        # Determine consistency level
        if consistency_rate == 1.0 or consistency_rate == 0.0:
            consistency = "CONSISTENT"
        elif consistency_rate >= 0.66:
            consistency = "MOSTLY_CONSISTENT"  
        else:
            consistency = "INCONSISTENT"
        
        all_results[(cause, effect)] = {
            'final_decision': final_valid,
            'consistency': consistency,
            'consistency_rate': consistency_rate,
            'avg_risk': avg_risk,
            'valid_runs': valid_count,
            'total_runs': n_runs,
            'method': method,
            'runs': run_results,
            'description': description
        }
        
        print(f"  → Final: {'VALID' if final_valid else 'REJECT'} | Consistency: {consistency} ({valid_count}/{n_runs})")
    
    return all_results


In [35]:
# Create hallbayes backend using the existing Azure OpenAI client
# We need to bypass the default OpenAI API key requirement
import os
from hallbayes import OpenAIBackend, OpenAIItem, OpenAIPlanner
temp_api_key = os.environ.get("PORTKEY_AZURE_US_API_KEY")

# Create the backend with our Azure model and then replace the client
hallbayes_backend = OpenAIBackend(model=azure_model, api_key=temp_api_key)

azure_openai_client = OpenAI(base_url=us_base_url,
            api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
            default_headers=portkey_headers)


# Replace the default client with our configured Azure OpenAI client 
hallbayes_backend.client = azure_openai_client

planner = OpenAIPlanner(hallbayes_backend, temperature=0.3)


In [ ]:
# # Test the unified function with professional prompts
# print("Testing unified validation function with professional prompts:")
# unified_results = validate_relationships_with_consistency(
#     relationships, planner, method="closed_book", threshold=0.10, n_runs=3
# )

Testing unified validation function with professional prompts:
Validating 3 relationships using CLOSED_BOOK method
Running 3 tests per relationship for consistency check

1/3: obesity → hypertension
  Run 1: VALID (Risk: 0.0%)
  Run 2: VALID (Risk: 0.0%)
  Run 3: VALID (Risk: 0.0%)
  → Final: VALID | Consistency: CONSISTENT (3/3)

2/3: age → obesity
  Run 1: REJECT (Risk: 100.0%)
  Run 2: REJECT (Risk: 100.0%)
  Run 3: REJECT (Risk: 100.0%)
  → Final: REJECT | Consistency: CONSISTENT (0/3)

3/3: age → hypertension
  Run 1: VALID (Risk: 0.0%)
  Run 2: VALID (Risk: 0.0%)
  Run 3: REJECT (Risk: 51.2%)
  → Final: VALID | Consistency: MOSTLY_CONSISTENT (2/3)


Test with complete KG

### Hallbayes validation with Evidence

In [10]:
# Example with new version
unified_results = validate_relationships_with_consistency(
    relationships, planner, method="with_evidence", threshold=0.10, n_runs=3
)

Validating 3 relationships using WITH_EVIDENCE method
Running 3 tests per relationship for consistency check

1/3: obesity → hypertension
  Run 1: REJECT (Risk: 0.0%)
  Run 2: REJECT (Risk: 0.0%)
  Run 3: REJECT (Risk: 0.0%)
  → Final: REJECT | Consistency: CONSISTENT (0/3)

2/3: age → obesity
  Run 1: REJECT (Risk: 100.0%)
  Run 2: REJECT (Risk: 100.0%)
  Run 3: VALID (Risk: 54.1%)
  → Final: REJECT | Consistency: INCONSISTENT (1/3)

3/3: age → hypertension
  Run 1: VALID (Risk: 0.0%)
  Run 2: VALID (Risk: 0.0%)
  Run 3: VALID (Risk: 0.0%)
  → Final: VALID | Consistency: CONSISTENT (3/3)


In [36]:
# all graph sale mal mejor closed book
unified_results = validate_relationships_with_consistency(
    results, planner, method="with_evidence", threshold=0.10, n_runs=3
)

Validating 19 relationships using WITH_EVIDENCE method
Running 3 tests per relationship for consistency check

1/19: bmi → systolic pressure (sysbp)
  Run 1: REJECT (Risk: 0.0%)
  Run 2: REJECT (Risk: 0.0%)
  Run 3: REJECT (Risk: 0.0%)
  → Final: REJECT | Consistency: CONSISTENT (0/3)

2/19: bmi → diastolic pressure (diabp)
  Run 1: REJECT (Risk: 0.0%)
  Run 2: REJECT (Risk: 0.0%)
  Run 3: REJECT (Risk: 0.0%)
  → Final: REJECT | Consistency: CONSISTENT (0/3)

3/19: age → bmi
  Run 1: REJECT (Risk: 0.0%)
  Run 2: REJECT (Risk: 0.0%)
  Run 3: REJECT (Risk: 0.0%)
  → Final: REJECT | Consistency: CONSISTENT (0/3)

4/19: bmi → cardiac ischemia
  Run 1: REJECT (Risk: 0.0%)
  Run 2: REJECT (Risk: 0.0%)
  Run 3: REJECT (Risk: 0.0%)
  → Final: REJECT | Consistency: CONSISTENT (0/3)

5/19: bmi → diabetes mellitus
  Run 1: REJECT (Risk: 0.0%)
  Run 2: REJECT (Risk: 0.0%)
  Run 3: REJECT (Risk: 0.0%)
  → Final: REJECT | Consistency: CONSISTENT (0/3)

6/19: bmi → major acute cardiovascular event
  

KeyboardInterrupt: 

it works with closed book better because we are not providing enough evidence here

In [ ]:
# Summary function for unified results
results = unified_results
def summarize_unified_results(unified_results):
    
    total = len(results)
    
    for (cause, effect), result in results.items():
        decision = "VALID" if result['final_decision'] else "REJECT"
        consistency = result['consistency']
        rate = result['consistency_rate']
        avg_risk = result['avg_risk']
        
        print(f"{cause} → {effect}:")
        print(f"  Decision: {decision} | Consistency: {consistency} ({rate:.1%}) | Avg Risk: {avg_risk:.1%}")
    
    print(f"\nRecommendation:")
    print("-" * 20)
    high_confidence = sum(1 for r in results.values() 
                         if r['final_decision'] and r['consistency'] == 'CONSISTENT')
    
    print(f"Use {high_confidence} relationships with VALID + CONSISTENT results for further analysis.")
    if high_confidence < total:
        uncertain = total - high_confidence
        print(f"Review {uncertain} relationships that are inconsistent or rejected.")

# Show summary of unified results
summarize_unified_results(results)

obesity → hypertension:
  Decision: VALID | Consistency: CONSISTENT (100.0%) | Avg Risk: 0.0%
age → obesity:
  Decision: REJECT | Consistency: CONSISTENT (0.0%) | Avg Risk: 83.7%
age → hypertension:
  Decision: VALID | Consistency: CONSISTENT (100.0%) | Avg Risk: 0.0%

Recommendation:
--------------------
Use 2 relationships with VALID + CONSISTENT results for further analysis.
Review 1 relationships that are inconsistent or rejected.


## Testing Logprobs Integration

Now let's test the updated method that can return logprobs data for confidence analysis.

In [5]:
# Test the updated method with logprobs
# First, let's reload the module to get the updated code

import importlib
import sys

# Remove the cached module if it exists
if 'pywhyllm.suggesters.simple_model_suggester' in sys.modules:
    importlib.reload(sys.modules['pywhyllm.suggesters.simple_model_suggester'])

from pywhyllm.suggesters.simple_model_suggester import SimpleModelSuggester

# Recreate the modeler with the updated class
modeler = SimpleModelSuggester(llm=model)
print("SimpleModelSuggester reloaded with logprobs support")

SimpleModelSuggester reloaded with logprobs support


In [ ]:
# Test without logprobs (original behavior)
result_simple = modeler.suggest_pairwise_relationship("smoking", "lung cancer")
print("Simple result:", result_simple)
print()

In [ ]:
# Test with logprobs enabled
result_with_logprobs = modeler.suggest_pairwise_relationship(
    "smoking", "lung cancer", return_logprobs=True
)

print("Result with logprobs:")
print("Causal relationship:", result_with_logprobs['result'])
print("Raw description:", result_with_logprobs['raw_description'][:200] + "...")
print("Logprobs data available:", result_with_logprobs['logprobs'] is not None)

if result_with_logprobs['logprobs']:
    print("Logprobs structure type:", type(result_with_logprobs['logprobs']))
    # Try to get confidence scores
    confidence = modeler.get_answer_confidence(result_with_logprobs['logprobs'])
    print("Confidence scores:", confidence)